# 01 Data Ingestion and Cleaning

This notebook loads OpenSky-style flights, inspects the incoming schema, normalizes the core columns, and writes the cleaned interim dataset used by the rest of the pipeline.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_data_paths, load_config
from src.clean_flights import clean_flights_df
from src.utils import read_table, setup_logging, write_table


In [ ]:
config = load_config(PROJECT_ROOT / 'configs' / 'default_config.yaml')
paths = get_data_paths(config, PROJECT_ROOT)
logger = setup_logging(config['logging']['level'])
raw_flights_path = paths['raw_dir'] / config['sources']['flights']['raw_filename']
raw_flights_path


In [ ]:
raw_df = read_table(raw_flights_path)
print(f'Raw rows: {len(raw_df):,}')
print('Columns:')
for column in raw_df.columns:
    print(f'  - {column}')
raw_df.head()


In [ ]:
clean_df, drop_stats = clean_flights_df(raw_df, logger)
print(f'Clean rows: {len(clean_df):,}')
drop_stats


In [ ]:
missing_summary = clean_df.isna().mean().sort_values(ascending=False).rename('missing_rate').to_frame()
missing_summary.head(10)


In [ ]:
clean_output = paths['interim_dir'] / 'flights_clean.parquet'
drop_stats_output = paths['interim_dir'] / 'flights_clean_drop_stats.csv'
write_table(clean_df, clean_output)
write_table(drop_stats, drop_stats_output)
print(clean_output)
clean_df.head()
